In [1]:
import pandas as pd
import numpy as np
import joblib

# Charger le modèle de classification (qu'on a sauvegardé)
# Si tu n'as pas encore fait le notebook 09, lance-le d'abord !
# Sinon, on peut continuer avec les variables du notebook 06

print("✅ Imports OK")

✅ Imports OK


C:\Users\suki\AppData\Local\Temp\ipykernel_24280\3079186718.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# Mots-clés qui indiquent une urgence dans le texte
MOTS_CRITIQUES = [
    'urgent', 'urgence', 'immediat', 'immediatement',
    'perissable', 'perissables', 'gate', 'gates', 'avarie',
    'medicament', 'medicaments', 'medical', 'medicale',
    'congele', 'congeles', 'frais', 'fraiche',
    'inadmissible', 'scandaleux', 'inacceptable',
    'avocat', 'avocats', 'plainte', 'tribunal', 'justice',
    'rembourser', 'remboursement', 'remboursez',
    'danger', 'dangereux', 'risque'
]

MOTS_ELEVE = [
    'tres', 'enorme', 'enormement', 'beaucoup',
    'jamais', 'totalement', 'completement', 'entierement',
    'furieux', 'furieuse', 'enerve', 'enervee', 'colere',
    'inadmissible', 'horrible', 'terrible', 'catastrophe',
    'mecontent', 'mecontente', 'decu', 'decue',
    'rapidement', 'vite', 'pressant'
]

MOTS_MOYEN = [
    'probleme', 'soucis', 'mauvais', 'mauvaise',
    'erreur', 'incorrect', 'incorrecte',
    'attendu', 'attendue', 'reception'
]

# Catégories qui sont généralement plus urgentes
CATEGORIES_URGENTES = {
    'produit_casse': 3,
    'retard_livraison': 2,
    'article_manquant': 2,
    'erreur_picking': 2,
    'probleme_transport': 2,
    'mauvaise_qualite': 1,
    'erreur_administrative': 1
}

# Types de produits avec leur urgence
PRODUITS_URGENCE = {
    'perissable': 5,
    'medicament': 5,
    'frais': 4,
    'fragile': 3,
    'electronique': 2,
    'standard': 0
}

print("✅ Règles de priorité définies")

✅ Règles de priorité définies


In [3]:
def calculer_priorite(texte, categorie, info_commande=None):
    """
    Calcule la priorité d'une réclamation selon plusieurs facteurs.
    
    Args:
        texte (str): Le texte de la réclamation (en français)
        categorie (str): La catégorie prédite par le modèle
        info_commande (dict): Optionnel, infos sur la commande
            {
                'type_produit': 'perissable' / 'standard' / etc.
                'client_prioritaire': True/False,
                'montant': 1500.0
            }
    
    Returns:
        str: 'faible', 'moyenne', 'elevee', ou 'critique'
        int: Le score calculé (pour debug)
    """
    score = 0
    raisons = []
    
    # 1. Analyse du texte (en minuscules, sans accents idéalement)
    texte_lower = str(texte).lower()
    
    # Compter les mots critiques
    nb_critiques = sum(1 for mot in MOTS_CRITIQUES if mot in texte_lower)
    nb_eleves = sum(1 for mot in MOTS_ELEVE if mot in texte_lower)
    nb_moyens = sum(1 for mot in MOTS_MOYEN if mot in texte_lower)
    
    score += nb_critiques * 4
    score += nb_eleves * 2
    score += nb_moyens * 1
    
    if nb_critiques > 0:
        raisons.append(f"{nb_critiques} mot(s) critique(s) détecté(s)")
    if nb_eleves > 0:
        raisons.append(f"{nb_eleves} mot(s) à forte urgence")
    
    # 2. Priorité selon la catégorie
    score_categorie = CATEGORIES_URGENTES.get(categorie, 0)
    score += score_categorie
    if score_categorie >= 2:
        raisons.append(f"Catégorie '{categorie}' généralement urgente")
    
    # 3. Infos sur la commande (si fournies)
    if info_commande:
        # Type de produit
        type_produit = info_commande.get('type_produit', 'standard').lower()
        score_produit = PRODUITS_URGENCE.get(type_produit, 0)
        score += score_produit
        if score_produit >= 3:
            raisons.append(f"Produit {type_produit} = haute urgence")
        
        # Client prioritaire
        if info_commande.get('client_prioritaire', False):
            score += 3
            raisons.append("Client prioritaire/VIP")
        
        # Montant élevé
        montant = info_commande.get('montant', 0)
        if montant > 5000:
            score += 3
            raisons.append(f"Montant élevé ({montant}€)")
        elif montant > 1000:
            score += 1
            raisons.append(f"Montant moyen ({montant}€)")
    
    # 4. Conversion score → niveau de priorité
    if score >= 10:
        priorite = 'critique'
    elif score >= 6:
        priorite = 'elevee'
    elif score >= 3:
        priorite = 'moyenne'
    else:
        priorite = 'faible'
    
    return priorite, score, raisons


# Test rapide
priorite, score, raisons = calculer_priorite(
    "C'est urgent ! Mon médicament n'est jamais arrivé, c'est inadmissible !",
    "retard_livraison",
    {'type_produit': 'medicament', 'client_prioritaire': True, 'montant': 50}
)

print(f"Priorité : {priorite}")
print(f"Score    : {score}")
print(f"Raisons  :")
for r in raisons:
    print(f"  - {r}")

Priorité : critique
Score    : 22
Raisons  :
  - 2 mot(s) critique(s) détecté(s)
  - 2 mot(s) à forte urgence
  - Catégorie 'retard_livraison' généralement urgente
  - Produit medicament = haute urgence
  - Client prioritaire/VIP


In [4]:
cas_tests = [
    {
        'texte': "Bonjour, ma commande est en retard de quelques jours, merci de verifier",
        'categorie': 'retard_livraison',
        'commande': {'type_produit': 'standard', 'montant': 50},
        'attendu': 'faible/moyenne'
    },
    {
        'texte': "URGENT ! Mes medicaments perissables ne sont jamais arrives, j'en ai besoin immediatement !",
        'categorie': 'retard_livraison',
        'commande': {'type_produit': 'medicament', 'client_prioritaire': True, 'montant': 200},
        'attendu': 'critique'
    },
    {
        'texte': "Le produit est arrive completement casse, c'est inadmissible !",
        'categorie': 'produit_casse',
        'commande': {'type_produit': 'fragile', 'montant': 800},
        'attendu': 'elevee'
    },
    {
        'texte': "La qualite n'est pas terrible, un peu decu",
        'categorie': 'mauvaise_qualite',
        'commande': {'type_produit': 'standard', 'montant': 30},
        'attendu': 'faible'
    },
    {
        'texte': "Il manque la moitie de ma commande, je veux un remboursement",
        'categorie': 'article_manquant',
        'commande': {'type_produit': 'standard', 'montant': 6000, 'client_prioritaire': True},
        'attendu': 'critique'
    },
]

print("=" * 75)
print("TEST DU MODULE DE PRIORITÉ")
print("=" * 75)

for i, cas in enumerate(cas_tests, 1):
    priorite, score, raisons = calculer_priorite(
        cas['texte'], cas['categorie'], cas['commande']
    )
    
    print(f"\n--- Cas {i} (attendu: {cas['attendu']}) ---")
    print(f"📝 Texte    : {cas['texte'][:80]}...")
    print(f"🏷️  Catégorie: {cas['categorie']}")
    print(f"📦 Produit  : {cas['commande'].get('type_produit', 'N/A')}")
    print(f"💰 Montant  : {cas['commande'].get('montant', 0)}€")
    print(f"⭐ PRIORITÉ : {priorite.upper()} (score: {score})")
    if raisons:
        print(f"💡 Raisons  : {', '.join(raisons[:3])}")

TEST DU MODULE DE PRIORITÉ

--- Cas 1 (attendu: faible/moyenne) ---
📝 Texte    : Bonjour, ma commande est en retard de quelques jours, merci de verifier...
🏷️  Catégorie: retard_livraison
📦 Produit  : standard
💰 Montant  : 50€
⭐ PRIORITÉ : FAIBLE (score: 2)
💡 Raisons  : Catégorie 'retard_livraison' généralement urgente

--- Cas 2 (attendu: critique) ---
📝 Texte    : URGENT ! Mes medicaments perissables ne sont jamais arrives, j'en ai besoin imme...
🏷️  Catégorie: retard_livraison
📦 Produit  : medicament
💰 Montant  : 200€
⭐ PRIORITÉ : CRITIQUE (score: 40)
💡 Raisons  : 7 mot(s) critique(s) détecté(s), 1 mot(s) à forte urgence, Catégorie 'retard_livraison' généralement urgente

--- Cas 3 (attendu: elevee) ---
📝 Texte    : Le produit est arrive completement casse, c'est inadmissible !...
🏷️  Catégorie: produit_casse
📦 Produit  : fragile
💰 Montant  : 800€
⭐ PRIORITÉ : CRITIQUE (score: 14)
💡 Raisons  : 1 mot(s) critique(s) détecté(s), 2 mot(s) à forte urgence, Catégorie 'produit_casse' génér

In [5]:
# On sauvegarde les règles dans un fichier Python qu'on importera dans l'API

code_priorite = '''
"""
Module de priorisation pour les réclamations logistiques.
"""

MOTS_CRITIQUES = [
    'urgent', 'urgence', 'immediat', 'immediatement',
    'perissable', 'perissables', 'gate', 'gates', 'avarie',
    'medicament', 'medicaments', 'medical', 'medicale',
    'congele', 'congeles', 'frais', 'fraiche',
    'inadmissible', 'scandaleux', 'inacceptable',
    'avocat', 'avocats', 'plainte', 'tribunal', 'justice',
    'rembourser', 'remboursement', 'remboursez',
    'danger', 'dangereux', 'risque'
]

MOTS_ELEVE = [
    'tres', 'enorme', 'enormement', 'beaucoup',
    'jamais', 'totalement', 'completement', 'entierement',
    'furieux', 'furieuse', 'enerve', 'enervee', 'colere',
    'inadmissible', 'horrible', 'terrible', 'catastrophe',
    'mecontent', 'mecontente', 'decu', 'decue',
    'rapidement', 'vite', 'pressant'
]

MOTS_MOYEN = [
    'probleme', 'soucis', 'mauvais', 'mauvaise',
    'erreur', 'incorrect', 'incorrecte',
    'attendu', 'attendue', 'reception'
]

CATEGORIES_URGENTES = {
    'produit_casse': 3,
    'retard_livraison': 2,
    'article_manquant': 2,
    'erreur_picking': 2,
    'probleme_transport': 2,
    'mauvaise_qualite': 1,
    'erreur_administrative': 1
}

PRODUITS_URGENCE = {
    'perissable': 5,
    'medicament': 5,
    'frais': 4,
    'fragile': 3,
    'electronique': 2,
    'standard': 0
}


def calculer_priorite(texte, categorie, info_commande=None):
    """Calcule la priorité d'une réclamation."""
    score = 0
    raisons = []
    
    texte_lower = str(texte).lower()
    
    nb_critiques = sum(1 for mot in MOTS_CRITIQUES if mot in texte_lower)
    nb_eleves = sum(1 for mot in MOTS_ELEVE if mot in texte_lower)
    nb_moyens = sum(1 for mot in MOTS_MOYEN if mot in texte_lower)
    
    score += nb_critiques * 4
    score += nb_eleves * 2
    score += nb_moyens * 1
    
    if nb_critiques > 0:
        raisons.append(f"{nb_critiques} mot(s) critique(s)")
    if nb_eleves > 0:
        raisons.append(f"{nb_eleves} mot(s) à forte urgence")
    
    score_categorie = CATEGORIES_URGENTES.get(categorie, 0)
    score += score_categorie
    if score_categorie >= 2:
        raisons.append(f"Categorie urgente")
    
    if info_commande:
        type_produit = info_commande.get('type_produit', 'standard').lower()
        score_produit = PRODUITS_URGENCE.get(type_produit, 0)
        score += score_produit
        if score_produit >= 3:
            raisons.append(f"Produit {type_produit}")
        
        if info_commande.get('client_prioritaire', False):
            score += 3
            raisons.append("Client prioritaire")
        
        montant = info_commande.get('montant', 0)
        if montant > 5000:
            score += 3
            raisons.append(f"Montant eleve")
        elif montant > 1000:
            score += 1
    
    if score >= 10:
        priorite = 'critique'
    elif score >= 6:
        priorite = 'elevee'
    elif score >= 3:
        priorite = 'moyenne'
    else:
        priorite = 'faible'
    
    return priorite, score, raisons
'''

# Sauvegarder dans src/
import os
os.makedirs('../src', exist_ok=True)
with open('../src/priority.py', 'w', encoding='utf-8') as f:
    f.write(code_priorite)

print("✅ Module sauvegardé dans : src/priority.py")

✅ Module sauvegardé dans : src/priority.py
